# Klein et al. (2017) – Droplet Barcoding for Single-Cell Transcriptomics Applied to Embryonic Stem Cells

The original paper can be found here: https://www.sciencedirect.com/science/article/pii/S0092867415005000

__Abstract__: It has long been the dream of biologists to map gene expression at the single-cell level. With such data one might track heterogeneous cell sub-populations, and infer regulatory relationships between genes and pathways. Recently, RNA sequencing has achieved single-cell resolution. What is limiting is an effective way to routinely isolate and process large numbers of individual cells for quantitative in-depth sequencing. We have developed a high-throughput droplet-microfluidic approach for barcoding the RNA from thousands of individual cells for subsequent analysis by next-generation sequencing. The method shows a surprisingly low noise profile and is readily adaptable to other sequencing-based assays. We analyzed mouse embryonic stem cells, revealing in detail the population structure and the heterogeneous onset of differentiation after leukemia inhibitory factor (LIF) withdrawal. The reproducibility of these high-throughput single-cell data allowed us to deconstruct cell populations and infer gene expression relationships.

Klein et al. analyze 2,717 single cells. The 4 classes in which they mention are experimental conditions/timepoints:

What the 4 classes are
- 0 days (baseline mESCs, +LIF)
- 2 days after LIF withdrawal
- 4 days after LIF withdrawal
- 7 days after LIF withdrawal

Klein profiled mESCs before and after LIF withdrawal to study early differentiation dynamics; GEO lists data for baseline ES cells and LIF− at days 2, 4, 7, which are the four-condition split many benchmarks (incl. scMINER) use.

---

## 0. Imports & Configuration

In [ ]:
from pathlib import Path

from benchmarks._studies import STUDIES, cvi_sweep, fit_or_load_carve, study_model_grids
from benchmarks.datasets import load_klein
from benchmarks.figures import (
    figure_carve_output_klein,
    figure_klein_results,
    prepare_composite,
)

RANDOM_SEED = 42
study = STUDIES["klein"]
model_grids = study_model_grids(study)

X, y, meta = load_klein(root=Path("../../data"), random_state=RANDOM_SEED)
print(f"{meta['n_cells']:,} cells x {meta['n_features']:,} genes")


---

## 3. Baseline Metrics

We evaluate Silhouette, Gap statistic, Davies-Bouldin, and Calinski-Harabasz across k = 2, ..., K for Agglomerative (ward) and Spectral clustering:

In [ ]:
curves_df, best_df = cvi_sweep(
    X, y, model_grids=model_grids, candidate_k=study.candidate_k,
    random_state=RANDOM_SEED, n_jobs=-1,
)
best_df


---

## 4. CARVE Analysis

In [ ]:
carve = fit_or_load_carve(
    X, y,
    cache_path=Path("./carve_state_saves/carve_klein.carve"),
    model_grids=model_grids,
    random_state=RANDOM_SEED,
)


---

## 5. Quantitative Comparison

### 5.1 Composite Paper Figure (with ARI Comparison)

In [ ]:
inputs = prepare_composite(
    X, y.to_numpy(), carve, curves_df=curves_df, best_df=best_df,
    comparison_metric="silhouette", random_state=RANDOM_SEED,
)
fig_output = figure_carve_output_klein(inputs)
fig_results = figure_klein_results(inputs)


---

## 6. Summary

Klein et al. is a droplet-based scRNA-seq benchmark with 4 experimental timepoints (d0, d2, d4, d7) tracking mESC differentiation after LIF withdrawal. The high-dimensional gene expression profiles and progressive differentiation dynamics provide a setting where the number of meaningful clusters is debatable. We compare CARVE's generalizability-based selection (measure=g, rule=1se, not_two=True) against four classical metrics and quantify agreement with the 4 reported timepoint labels using ARI.

---